In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

# --------------------------------------------------
# 1. Tiny toy translation dataset
# --------------------------------------------------

pairs = [
    # Pronouns + être (to be)
    ("i am happy", "je suis heureux"),
    ("i am sad", "je suis triste"),
    ("i am tired", "je suis fatigue"),
    ("you are happy", "tu es heureux"),
    ("you are sad", "tu es triste"),
    ("he is happy", "il est heureux"),
    ("she is happy", "elle est heureuse"),

    # More basic sentences
    ("i am ready", "je suis pret"),
    ("i am here", "je suis ici"),
    ("you are here", "tu es ici"),
    ("he is here", "il est ici"),
    ("she is here", "elle est ici"),
    ("we are happy", "nous sommes heureux"),
    ("they are happy", "ils sont heureux"),
    ("they are here", "ils sont ici"),
]

pairs += [
    ("i eat bread", "je mange du pain"),
    ("i drink water", "je bois de l eau"),
    ("i like music", "j aime la musique"),
    ("i like coffee", "j aime le cafe"),
    ("i love cats", "j aime les chats"),
    ("i love dogs", "j aime les chiens"),
    ("you eat bread", "tu manges du pain"),
    ("you drink water", "tu bois de l eau"),
    ("he eats bread", "il mange du pain"),
    ("she drinks water", "elle boit de l eau"),
    ("we eat together", "nous mangeons ensemble"),
    ("we drink coffee", "nous buvons du cafe"),
    ("they like music", "ils aiment la musique"),
    ("they eat together", "ils mangent ensemble"),
]

pairs += [
    ("good morning", "bonjour"),
    ("good evening", "bonsoir"),
    ("good night", "bonne nuit"),
    ("thank you", "merci"),
    ("thank you very much", "merci beaucoup"),
    ("see you soon", "a bientot"),
    ("see you tomorrow", "a demain"),
    ("i am hungry", "j ai faim"),
    ("i am thirsty", "j ai soif"),
    ("i am cold", "j ai froid"),
    ("i am hot", "j ai chaud"),
    ("i am afraid", "j ai peur"),
    ("i have a book", "j ai un livre"),
    ("i have a cat", "j ai un chat"),
    ("i have a dog", "j ai un chien"),
]


pairs += [
    ("how are you", "comment vas tu"),
    ("what is your name", "comment tu t appelles"),
    ("where are you", "ou es tu"),
    ("where is he", "ou est il"),
    ("where is she", "ou est elle"),
    ("are you happy", "es tu heureux"),
    ("are you ready", "es tu pret"),
    ("do you like coffee", "aimes tu le cafe"),
    ("do you like cats", "aimes tu les chats"),
    ("what do you want", "que veux tu"),
    ("what are you doing", "que fais tu"),
    ("where do you live", "ou habites tu"),
]

source_special = ["<pad>", "<unk>"]
target_special = ["<pad>", "<unk>", "<sos>", "<eos>"]

source_words = sorted(set(
    word for src, _ in pairs for word in src.split()
))
target_words = sorted(set(
    word for _, tgt in pairs for word in tgt.split()
))



In [14]:
source_words

['a',
 'afraid',
 'am',
 'are',
 'book',
 'bread',
 'cat',
 'cats',
 'coffee',
 'cold',
 'do',
 'dog',
 'dogs',
 'doing',
 'drink',
 'drinks',
 'eat',
 'eats',
 'evening',
 'good',
 'happy',
 'have',
 'he',
 'here',
 'hot',
 'how',
 'hungry',
 'i',
 'is',
 'like',
 'live',
 'love',
 'morning',
 'much',
 'music',
 'name',
 'night',
 'ready',
 'sad',
 'see',
 'she',
 'soon',
 'thank',
 'they',
 'thirsty',
 'tired',
 'together',
 'tomorrow',
 'very',
 'want',
 'water',
 'we',
 'what',
 'where',
 'you',
 'your']

In [15]:
target_words

['a',
 'ai',
 'aime',
 'aiment',
 'aimes',
 'appelles',
 'beaucoup',
 'bientot',
 'bois',
 'boit',
 'bonjour',
 'bonne',
 'bonsoir',
 'buvons',
 'cafe',
 'chat',
 'chats',
 'chaud',
 'chien',
 'chiens',
 'comment',
 'de',
 'demain',
 'du',
 'eau',
 'elle',
 'ensemble',
 'es',
 'est',
 'faim',
 'fais',
 'fatigue',
 'froid',
 'habites',
 'heureuse',
 'heureux',
 'ici',
 'il',
 'ils',
 'j',
 'je',
 'l',
 'la',
 'le',
 'les',
 'livre',
 'mange',
 'mangent',
 'mangeons',
 'manges',
 'merci',
 'musique',
 'nous',
 'nuit',
 'ou',
 'pain',
 'peur',
 'pret',
 'que',
 'soif',
 'sommes',
 'sont',
 'suis',
 't',
 'triste',
 'tu',
 'un',
 'vas',
 'veux']

In [16]:
src_vocab = {
    word: i for i, word in enumerate(source_special + source_words)
}
tgt_vocab = {
    word: i for i, word in enumerate(target_special + target_words)
}

tgt_idx_to_word = {i: w for w, i in tgt_vocab.items()}



In [17]:
src_vocab

{'<pad>': 0,
 '<unk>': 1,
 'a': 2,
 'afraid': 3,
 'am': 4,
 'are': 5,
 'book': 6,
 'bread': 7,
 'cat': 8,
 'cats': 9,
 'coffee': 10,
 'cold': 11,
 'do': 12,
 'dog': 13,
 'dogs': 14,
 'doing': 15,
 'drink': 16,
 'drinks': 17,
 'eat': 18,
 'eats': 19,
 'evening': 20,
 'good': 21,
 'happy': 22,
 'have': 23,
 'he': 24,
 'here': 25,
 'hot': 26,
 'how': 27,
 'hungry': 28,
 'i': 29,
 'is': 30,
 'like': 31,
 'live': 32,
 'love': 33,
 'morning': 34,
 'much': 35,
 'music': 36,
 'name': 37,
 'night': 38,
 'ready': 39,
 'sad': 40,
 'see': 41,
 'she': 42,
 'soon': 43,
 'thank': 44,
 'they': 45,
 'thirsty': 46,
 'tired': 47,
 'together': 48,
 'tomorrow': 49,
 'very': 50,
 'want': 51,
 'water': 52,
 'we': 53,
 'what': 54,
 'where': 55,
 'you': 56,
 'your': 57}

In [18]:
tgt_vocab

{'<pad>': 0,
 '<unk>': 1,
 '<sos>': 2,
 '<eos>': 3,
 'a': 4,
 'ai': 5,
 'aime': 6,
 'aiment': 7,
 'aimes': 8,
 'appelles': 9,
 'beaucoup': 10,
 'bientot': 11,
 'bois': 12,
 'boit': 13,
 'bonjour': 14,
 'bonne': 15,
 'bonsoir': 16,
 'buvons': 17,
 'cafe': 18,
 'chat': 19,
 'chats': 20,
 'chaud': 21,
 'chien': 22,
 'chiens': 23,
 'comment': 24,
 'de': 25,
 'demain': 26,
 'du': 27,
 'eau': 28,
 'elle': 29,
 'ensemble': 30,
 'es': 31,
 'est': 32,
 'faim': 33,
 'fais': 34,
 'fatigue': 35,
 'froid': 36,
 'habites': 37,
 'heureuse': 38,
 'heureux': 39,
 'ici': 40,
 'il': 41,
 'ils': 42,
 'j': 43,
 'je': 44,
 'l': 45,
 'la': 46,
 'le': 47,
 'les': 48,
 'livre': 49,
 'mange': 50,
 'mangent': 51,
 'mangeons': 52,
 'manges': 53,
 'merci': 54,
 'musique': 55,
 'nous': 56,
 'nuit': 57,
 'ou': 58,
 'pain': 59,
 'peur': 60,
 'pret': 61,
 'que': 62,
 'soif': 63,
 'sommes': 64,
 'sont': 65,
 'suis': 66,
 't': 67,
 'triste': 68,
 'tu': 69,
 'un': 70,
 'vas': 71,
 'veux': 72}

In [19]:
tgt_idx_to_word

{0: '<pad>',
 1: '<unk>',
 2: '<sos>',
 3: '<eos>',
 4: 'a',
 5: 'ai',
 6: 'aime',
 7: 'aiment',
 8: 'aimes',
 9: 'appelles',
 10: 'beaucoup',
 11: 'bientot',
 12: 'bois',
 13: 'boit',
 14: 'bonjour',
 15: 'bonne',
 16: 'bonsoir',
 17: 'buvons',
 18: 'cafe',
 19: 'chat',
 20: 'chats',
 21: 'chaud',
 22: 'chien',
 23: 'chiens',
 24: 'comment',
 25: 'de',
 26: 'demain',
 27: 'du',
 28: 'eau',
 29: 'elle',
 30: 'ensemble',
 31: 'es',
 32: 'est',
 33: 'faim',
 34: 'fais',
 35: 'fatigue',
 36: 'froid',
 37: 'habites',
 38: 'heureuse',
 39: 'heureux',
 40: 'ici',
 41: 'il',
 42: 'ils',
 43: 'j',
 44: 'je',
 45: 'l',
 46: 'la',
 47: 'le',
 48: 'les',
 49: 'livre',
 50: 'mange',
 51: 'mangent',
 52: 'mangeons',
 53: 'manges',
 54: 'merci',
 55: 'musique',
 56: 'nous',
 57: 'nuit',
 58: 'ou',
 59: 'pain',
 60: 'peur',
 61: 'pret',
 62: 'que',
 63: 'soif',
 64: 'sommes',
 65: 'sont',
 66: 'suis',
 67: 't',
 68: 'triste',
 69: 'tu',
 70: 'un',
 71: 'vas',
 72: 'veux'}

In [20]:

PAD = tgt_vocab["<pad>"]
SOS = tgt_vocab["<sos>"]
EOS = tgt_vocab["<eos>"]

# --------------------------------------------------
# 2. Encoder
# --------------------------------------------------

class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=16, hidden_size=32):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(
            embedding_dim, hidden_size, batch_first=True
        )

    def forward(self, src):
        # src:       [B, S]
        embedded = self.embedding(src)
        # embedded:  [B, S, E]

        outputs, hidden = self.rnn(embedded)
        # outputs:   [B, S, H]
        # hidden:    [1, B, H]

        return outputs, hidden


# --------------------------------------------------
# 3. Decoder
# --------------------------------------------------

class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=16, hidden_size=32):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(
            embedding_dim, hidden_size, batch_first=True
        )
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, token, hidden):
        # token:     [B, 1]
        embedded = self.embedding(token)
        # embedded:  [B, 1, E]

        output, hidden = self.rnn(embedded, hidden)
        # output:    [B, 1, H]
        # hidden:    [1, B, H]

        logits = self.fc(output)
        # logits:    [B, 1, target_vocab_size]

        return logits, hidden


# --------------------------------------------------
# 4. Combine encoder and decoder
# --------------------------------------------------

class Seq2Seq(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size):
        super().__init__()

        self.encoder = Encoder(src_vocab_size)
        self.decoder = Decoder(tgt_vocab_size)

    def forward(self, src, decoder_input):
        _, encoder_hidden = self.encoder(src)

        hidden = encoder_hidden
        logits_list = []

        # Training: feed each correct previous target token.
        for t in range(decoder_input.size(1)):
            token = decoder_input[:, t:t+1]
            logits, hidden = self.decoder(token, hidden)
            logits_list.append(logits)

        return torch.cat(logits_list, dim=1)

In [21]:
source_vocab_size = len(src_vocab)
target_vocab_size = len(tgt_vocab)
source_vocab_size, target_vocab_size

(58, 73)

In [25]:
# --------------------------------------------------
# 5. Training
# --------------------------------------------------

model = Seq2Seq(source_vocab_size, target_vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10000):
    total_loss = 0

    for src_text, tgt_text in pairs:
        src_ids = [src_vocab[w] for w in src_text.split()]

        tgt_ids = (
            [SOS]
            + [tgt_vocab[w] for w in tgt_text.split()]
            + [EOS]
        )

        src = torch.tensor([src_ids], dtype=torch.long)

        # Inputs: SOS, Je, mange, riz
        decoder_input = torch.tensor(
            [tgt_ids[:-1]], dtype=torch.long
        )

        # Labels: Je, mange, riz, EOS
        target = torch.tensor(
            [tgt_ids[1:]], dtype=torch.long
        )

        optimizer.zero_grad()

        logits = model(src, decoder_input)

        loss = F.cross_entropy(
            logits.reshape(-1, len(tgt_vocab)),
            target.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 100 == 0:
        print(
            f"Epoch {epoch + 1}, "
            f"loss = {total_loss / len(pairs):.4f}"
        )

Epoch 100, loss = 0.0759
Epoch 200, loss = 0.0042
Epoch 300, loss = 0.0012
Epoch 400, loss = 0.0001


KeyboardInterrupt: 

In [26]:
@torch.no_grad()
def translate(model, sentence, max_length=10):
    model.eval()

    # 1. Convert source words to token IDs.
    words = sentence.lower().split()

    src_ids = [
        src_vocab.get(word, src_vocab["<unk>"])
        for word in words
    ]

    src = torch.tensor([src_ids], dtype=torch.long)

    # 2. Encode the source sentence.
    _, hidden = model.encoder(src)

    # 3. Start the decoder with SOS.
    token = torch.tensor([[SOS]], dtype=torch.long)

    result = []

    # 4. Generate one token at a time.
    for _ in range(max_length):
        logits, hidden = model.decoder(token, hidden)

        # Choose the highest-scoring token.
        next_id = logits[0, 0].argmax().item()

        # Stop if the model predicts EOS.
        if next_id == EOS:
            break

        # Convert token ID to a word.
        result.append(tgt_idx_to_word[next_id])

        # Feed the prediction into the next step.
        token = torch.tensor([[next_id]], dtype=torch.long)

    return " ".join(result)

In [30]:
print(translate(model, "i love bread"))


j aime les chiens


In [ ]:
pairs += [
    ("i eat bread", "je mange du pain"),
    ("i drink water", "je bois de l eau"),
    ("i like music", "j aime la musique"),
    ("i like coffee", "j aime le cafe"),
    ("i love cats", "j aime les chats"),
    ("i love dogs", "j aime les chiens"),
    ("you eat bread", "tu manges du pain"),
    ("you drink water", "tu bois de l eau"),
    ("he eats bread", "il mange du pain"),
    ("she drinks water", "elle boit de l eau"),
    ("we eat together", "nous mangeons ensemble"),
    ("we drink coffee", "nous buvons du cafe"),
    ("they like music", "ils aiment la musique"),
    ("they eat together", "ils mangent ensemble"),
]